<div style="font-size: 0.6em; max-width: 70ch">

# Importing packages

</div>

In [1]:
import pandas as pd
import plotly.express as px

# To display the figure defined by this dict, use the low-level 
# plotly.io.show function.
import plotly.io as pio 
# Default is plotly_mimetype+notebook, but jekyll fails to 
# parse plotly_mimetype.
pio.renderers.default = 'notebook_connected'
# Inject the missing require.js dependency.
from IPython.display import display, HTML
js = '<script src="https://cdnjs.cloudflare.com/ajax/libs/require.js' \
'/2.3.6/require.min.js" integrity="sha512-c3Nl8+7g4LMSTdrm621y7kf9v3SDP' \
'nhxLNhcjFJbKECVnmZHTdo+IRO05sNLTH/D3vA6u1X32ehoLC7WFVdheg==" ' \
'crossorigin="anonymous"></script>'
display(HTML(js))

import ardalan_proEV_functions as AF
# Import the custom functions for this notebook.
import cellgrowth.core as cg

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

<div style="font-size: 0.6em; max-width: 100ch">

# Importing and cleaning of the primary raw data

</div>

In [2]:
# Read the data file, skipping the first 8 rows of metadata.
df = pd.read_csv(r"../data/cell_growth_data.txt", 
                 delimiter = '\t', skiprows=8)

# Make a plate format inducating the row and column of the sample
# Map the numeric row values to letters for plate format.
cg.map_num_to_letter(df)

# Create a 'Plate format' column by concatenating the 
# 'Row' and 'Column' values.
df.Column = df.Column.astype(str)
df['Plate format'] = df[['Row', 'Column']].apply(lambda x: ''.join(x), axis=1)

In [3]:
df.head(1)

,Row,Column,Timepoint,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s],Temperature,Target Temperature,CO2,Target CO2,Compound,Concentration,Cell Type,Cell Count,Unnamed: 19,Plate format
0,E,4,0,770,302.658182,0.886863,2030.238556,1.044651e+07,768244.360521,25,0.0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,E4


In [4]:
df.columns.to_list()

['Row',
 'Column',
 'Timepoint',
 'Cells Selected - Number of Objects',
 'Cells Selected - Cell Area [µm²] - Mean per Well',
 'Cells Selected - Cell Roundness - Mean per Well',
 'Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well',
 'Image Region - Image Region Area [µm²] - Sum per Well',
 'Texture A Selected - Region Area [µm²] - Sum per Well',
 'Number of Analyzed Fields',
 'Time [s]',
 'Temperature',
 'Target Temperature',
 'CO2',
 'Target CO2',
 'Compound',
 'Concentration',
 'Cell Type',
 'Cell Count',
 'Unnamed: 19',
 'Plate format']

In [5]:
df = df [['Row',
 'Column',
 'Timepoint',
  'Plate format',
 'Cells Selected - Number of Objects',
 'Cells Selected - Cell Area [µm²] - Mean per Well',
 'Cells Selected - Cell Roundness - Mean per Well',
 'Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well',
 'Image Region - Image Region Area [µm²] - Sum per Well',
 'Texture A Selected - Region Area [µm²] - Sum per Well',
 'Number of Analyzed Fields',
 'Time [s]',
]]

<div style="font-size: 0.6em; max-width: 70ch">

# Data visualization

</div>

In [6]:
df.head(2)

,Row,Column,Timepoint,Plate format,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s]
0,E,4,0,E4,770,302.658182,0.886863,2030.238556,1.044651e+07,768244.360521,25,0.0
1,E,5,0,E5,881,322.810309,0.881544,2048.947241,1.044809e+07,701412.476601,25,0.0


In [7]:
df["Timepoint"].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
      dtype=int64)

<div style="font-size: 0.6em; max-width: 70ch">

# Defining the samples of each well

</div>

In [8]:
df.loc[df['Plate format'].isin(['E4', 'E5']), ['Sample']] = 'Neg Control'

df.loc[df['Plate format'].isin(['F4', 'F5']), ['Sample']] = 'Parental'
 
df.loc[df['Plate format'].isin(['G4', 'G5']), ['Sample']] = 'LOW_uptake'
 
df.loc[df['Plate format'].isin(['H4', 'H5']), ['Sample']] = 'HIGH_uptake'

<div style="font-size: 0.6em; max-width: 100ch">

# Preparing dataframe for making lineplots

</div>

* As each sample is in duplicate, for making a line graph, one spot per time-point makes more sense. For that, I calculated the average of each
duplicate value using groupby. 

In [9]:
data = cg.DataProcessing(df)

# Process the timepoints to get a DataFrame with average time in seconds and 
# update the df inplace.  
# for each timepoint. This will be used for plotting the growth curves with 
# time on the x-axis.
timing = data.process_timepoints(timepoint_col='Timepoint', time_col='Time [s]')

# Generate a grouped DataFrame for plotting growth curves.
df_grouped = data.grouped()
df_grouped.head()

,Sample,Timepoint,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s],Hours,cell_covered_area_std,cell_count_std
0,HIGH_uptake,0,1000.5,325.879734,0.872133,2297.879871,1.044785e+07,8.413457e+05,25.0,0.000,4.0,17480.305526,54.447222
1,HIGH_uptake,1,1056.5,374.740170,0.877443,2415.943798,1.044769e+07,1.087237e+06,25.0,21564.770,10.0,20712.256734,19.091883
2,HIGH_uptake,2,1032.0,480.300880,0.899131,2405.490523,1.044779e+07,1.444340e+06,25.0,43127.292,16.0,55226.688251,28.284271
3,HIGH_uptake,3,1180.0,542.115886,0.908378,2204.343429,1.044805e+07,1.935250e+06,25.0,70635.050,24.0,42149.396835,66.468037
4,HIGH_uptake,4,1426.0,544.635183,0.903243,2097.330607,1.044804e+07,2.314716e+06,25.0,91407.058,29.0,105368.804793,70.710678


"""Sort the samples for plotting:"""

In [10]:
# First check the order of the sample in the df, which determines their order on the graph
df_grouped["Sample"].unique()

array(['HIGH_uptake', 'LOW_uptake', 'Neg Control', 'Parental'],
      dtype=object)

In [11]:
# Sort the sample names as you wish
sample_order = ['Neg Control', "Parental", 'LOW_uptake', 'HIGH_uptake']
# During the sorting, sorting the order of the values too. In this experiment, 
# the order of the hours should be considered too. 
# Sort based on your custom sample order and the order of hours (ascending)
df_grouped = cg.sample_order_sorter(
    df_grouped, sample_order, "Sample", ["Hours"]
)
df_grouped["Sample"].unique()

array(['Neg Control', 'Parental', 'LOW_uptake', 'HIGH_uptake'],
      dtype=object)

<div style="font-size: 0.6em; max-width: 200ch">

# Lineplot showing the cell growth over time_ covered cell area

</div>

In [13]:
df_grouped.head()

,Sample,Timepoint,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s],Hours,cell_covered_area_std,cell_count_std,Samples_order
28,Neg Control,0,825.5,312.734245,0.884203,2039.592899,1.044730e+07,7.348284e+05,25.0,0.0000,4.0,47257.278319,78.488853,1
29,Neg Control,1,837.5,338.889273,0.893481,2067.363578,1.044693e+07,8.247876e+05,25.0,21564.6935,10.0,88177.844538,3.535534,1
30,Neg Control,2,868.5,386.446302,0.900810,1989.935957,1.044721e+07,9.380410e+05,25.0,43127.1585,16.0,113061.537812,24.748737,1
31,Neg Control,3,1014.0,415.559236,0.895801,1794.238645,1.044791e+07,1.080807e+06,25.0,70635.2250,24.0,90373.017414,19.798990,1
32,Neg Control,4,1226.0,416.699435,0.896083,1743.483731,1.044788e+07,1.320422e+06,25.0,91407.2465,29.0,115943.842361,42.426407,1


In [ ]:
# usage 
graph = cg.Graph()
fig = graph.line_graph(
    df_grouped,
    df_grouped,
    x="Hours",  
    y="Texture A Selected - Region Area [µm²] - Sum per Well", 
    color="Sample", 
    color_discrete_map={
        "Neg Control": "black",
        "Parental": "green" , 
        "LOW_uptake": "blue", 
        "HIGH_uptake": "red"
    }
)

graph.update_parameters(
    yaxis={"title_text": "Plate-well surface area covered by cell",},
    layout={"title": {"text": "Cell growth over time for low-uptake and high-uptake_Exp. 4"}},
)
fig.show()

# Export the graph
# fig.write_html(r"YOUR_EXPORT_PATH\figure_title.html")

<div style="font-size: 0.6em; max-width: 100ch">

# Ratio of cell growth_ based on cell covered area

</div>

In [15]:
ratios_list = []
for group in grouped_df["Sample"].unique():
    data = grouped_df.loc[grouped_df["Sample"] == group]
    data = data.reset_index()
    series = data["Texture A Selected - Region Area [µm²] - Sum per Well"]
    ratios = [series[i] / series[0] for i, value in enumerate(series) if i > 0] # Every values, except the first value (timepoint 0), is divided by timepoint zero. to \
                                                                                # find the ratio. This is a way of normalization.  
    ratios.insert(0, 1) # The first values would be 1   
    data["Ratios"] = ratios
    ratios_list.append(data)

grouped_df = pd.concat(ratios_list)    

In [16]:
fig = px.line(grouped_df, x="Hours" , y="Ratios", color="Sample", 
             color_discrete_map={"Neg Control": "black", "Parental": "green", "LOW_uptake": "blue", "HIGH_uptake": "red"},  
             hover_data=["Texture A Selected - Region Area [µm²] - Sum per Well", "Timepoint"], markers=True,
              width=1200, height=600)
 
# Make a box around the figure and define the range of x and y axes.
ticks = grouped_df["Hours"].to_numpy() # use the exact hours for the x axis 
fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", 
                 title_text="Hours", tickvals=ticks, range=(0, 100), tickfont_size=20, title_font_size=20)
fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", title_text="Growth ratios_Compared to timepoint 0", tickfont_size=20, title_font_size=20)

fig.update_layout(plot_bgcolor="white", title=dict(text="Growth ratio over time_based on cell covered area_Exp. 4", font_size=22, x=0.5))

fig.update_traces(textposition="top center") # If you have selected some text for the data point, you can define their positions. 

fig.show()

<div style="font-size: 0.6em; max-width: 100ch">

# Lineplot showing the cell growth over time_ absolute cell count using DPC

</div>

In [17]:
fig = px.line(grouped_df, x="Hours" , y="Cells Selected - Number of Objects", color="Sample", 
             color_discrete_map={"Neg Control": "black", "Parental": "green", "LOW_uptake": "blue", "HIGH_uptake": "red"},  
             hover_data=["Texture A Selected - Region Area [µm²] - Sum per Well", "Timepoint"], markers=True,
             error_y="cell_count_std", width=1200, height=600)
 
# Make a box around the figure and define the range of x and y axes.
ticks = grouped_df["Hours"].to_numpy() # use the exact hours for the x axis 
fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", 
                 title_text="Hours", tickvals=ticks, range=(0, 100), tickfont_size=20, title_font_size=20)
fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", title_text="Cell count", tickfont_size=20, title_font_size=20)

fig.update_layout(plot_bgcolor="white", title=dict(text="Cell count over time for low-uptake and high-uptake_Exp. 4", font_size=26, x=0.5))

fig.update_traces(textposition="top center") # If you have selected some text for the data point, you can define their positions. 

# Add the button for turning the error-bar display on and off. 
updatemenus = [{
                'active':0,
                'buttons': [{'method': 'update',
                             'label': 'error-bar display',
                             'args': [
                                      # 1. updates to the traces
                                     
                                      {'error_y': [dict(type="grouped_df", array=grouped_df["cell_count_std"].loc[grouped_df["Sample"] == "Neg Control"]),
                                                  dict(type="grouped_df", array=grouped_df["cell_count_std"].loc[grouped_df["Sample"] == "Parental"]),
                                                  dict(type="grouped_df", array=grouped_df["cell_count_std"].loc[grouped_df["Sample"] == "LOW_uptake"]),
                                                  dict(type="grouped_df", array=grouped_df["cell_count_std"].loc[grouped_df["Sample"] == "HIGH_uptake"])
                                                  ]}, 
                                    ],
                             'args2': [
                                       # 1. updates to the traces  
                                       {'error_y': dict(type="data", array=[])},
                                      ],
                             'visible':True}
                            
                           ],
                'type':'buttons',
                'showactive': True,
                'x': 1.15,
                'y': 0.7,
                'xanchor': "right", 
                'yanchor': "top"
                },]


fig.update_layout(updatemenus=updatemenus)

fig.show()


<div style="font-size: 0.6em; max-width: 100ch">

# Ratio of cell growth_ based on DPC cell count

</div>

In [18]:
grouped_df.head()

,index,Sample,Timepoint,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s],Hours,cell_covered_area_std,cell_count_std,Samples_order,Ratios
0,28,Neg Control,0,825.5,312.734245,0.884203,2039.592899,1.044730e+07,7.348284e+05,25.0,0.0000,4.0,47257.278319,78.488853,1,1.000000
1,29,Neg Control,1,837.5,338.889273,0.893481,2067.363578,1.044693e+07,8.247876e+05,25.0,21564.6935,10.0,88177.844538,3.535534,1,1.122422
2,30,Neg Control,2,868.5,386.446302,0.900810,1989.935957,1.044721e+07,9.380410e+05,25.0,43127.1585,16.0,113061.537812,24.748737,1,1.276544
3,31,Neg Control,3,1014.0,415.559236,0.895801,1794.238645,1.044791e+07,1.080807e+06,25.0,70635.2250,24.0,90373.017414,19.798990,1,1.470829
4,32,Neg Control,4,1226.0,416.699435,0.896083,1743.483731,1.044788e+07,1.320422e+06,25.0,91407.2465,29.0,115943.842361,42.426407,1,1.796913


In [19]:
ratios_list = []
# Remove the index column made during the grouping for the cell surface area before. 
#grouped_df.drop(columns="level_0", inplace=True)
for group in grouped_df["Sample"].unique():
    dataframe = grouped_df.loc[grouped_df["Sample"] == group]
    dataframe = dataframe.reset_index()
    series = dataframe["Cells Selected - Number of Objects"]
    ratios = [series[i] / series[0] for i, value in enumerate(series) if i > 0] # Every values, except the first value (timepoint 0), is divided by timepoint zero. to \
                                                                                # find the ratio. This is a way of normalization.  
    ratios.insert(0, 1) # The first values would be 1   
    dataframe["Ratios_cell_count"] = ratios
    ratios_list.append(dataframe)

df_ratios = pd.concat(ratios_list)    

In [20]:
fig = px.line(df_ratios, x="Hours" , y="Ratios_cell_count", color="Sample", 
             color_discrete_map={"Neg Control": "black", "Parental": "green", "LOW_uptake": "blue", "HIGH_uptake": "red"},  
             hover_data=["Texture A Selected - Region Area [µm²] - Sum per Well", "Timepoint"], markers=True,
             width=1200, height=400)
 
# Make a box around the figure and define the range of x and y axes.
ticks = grouped_df["Hours"].to_numpy() # use the exact hours for the x axis 
fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", 
                 title_text="Hours", tickvals=ticks, range=(0, 100), tickfont_size=35, title_font_size=40)
fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", title_text="Growth <br> fold change", tickfont_size=35, title_font_size=40)

fig.update_layout(plot_bgcolor="white", title=dict(text="Growth curve for experiment 20Nov2023", x=0.5), 
font_size=22)

fig.update_traces(textposition="top center") # If you have selected some text for the data point, you can define their positions. 

#fig.show()




<div style="font-size: 0.6em; max-width: 70ch">

# Normalized DPC for poster

</div>

In [21]:
df_ratios_poster = df_ratios.loc[~ (df_ratios["Hours"] > 72)]
df_ratios_poster = df_ratios_poster.loc[~(df_ratios_poster["Sample"] == "Parental")]
df_ratios_poster["Sample"] = ["Control" if x == "Neg Control" else x for x in df_ratios_poster["Sample"].values]
fig = px.line(df_ratios_poster, x="Hours" , y="Ratios_cell_count", color="Sample", 
             color_discrete_map={"Control": "black", "LOW_uptake": "blue", "HIGH_uptake": "red"},  
             hover_data=["Texture A Selected - Region Area [µm²] - Sum per Well", "Timepoint"], markers=True,
             width=1200, height=600)
 
# Make a box around the figure and define the range of x and y axes.
ticks = grouped_df["Hours"].to_numpy() # use the exact hours for the x axis 
fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", 
                 title_text="Hours", tickvals=ticks, range=(0, 75), tickfont_size=35, title_font_size=40)
fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", title_text="Growth <br> fold change", 
                 tickfont_size=35, title_font_size=40)

fig.update_layout(plot_bgcolor="white", title=dict(text="Growth curve for experiment 19oct2023", x=0.5), 
font_size=22, height= 350, width= 1000)

fig.update_traces(textposition="top center") # If you have selected some text for the data point, you can define their positions. 
fig.update_layout(showlegend=True) # Turn off the legend

#fig.show()




<div style="font-size: 0.6em; max-width: 70ch">

# Scatterplot, as an alternative to histogram

</div>

In [22]:
df.head(1)

,Row,Column,Timepoint,Plate format,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s],Sample,Hours,cell_covered_area_std,cell_count_std
0,E,4,0,E4,770,302.658182,0.886863,2030.238556,1.044651e+07,768244.360521,25,0.0,Neg Control,4,768244.360521,770


In [23]:
for exp in df["Experimental_ID"].unique():
    exp_df = df.loc[df["Experimental_ID"] == exp]
    fig = px.scatter(exp_df, x="Size_normalized_Comp-YG-PE-A :: PKH26" , y="SSC-H" , color="Sample", 
                opacity=0.5, 
                category_orders= {"Sample": ["High", "Low", "Control", ]}, color_discrete_sequence= ["red", "blue", "black"])
    
    # Make a box around the figure and define the range of x and y axes.
    fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", range=[0, 300000], title_text="PKH26 signal intensity")
    fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", range=[0, 250000], title_text="SSC-A")
 
    
    # Add title to each set of subplot and change the background color to white and adjust the size of the fig. 
    date = exp_df["Date_of_experiment"].unique()[0]
    title = "Sorting " + str(exp) + "<br>" + date    # Make the title as the name of the experiment
    fig.update_layout(title= title, title_x=0.5) # Add title and place it in the middle of the canvas
    fig.update_layout(height=320, width=800, plot_bgcolor="white")
    
    # Export the figs
    exp_name = "Sorting " + str(exp) + "_" + date # You can not use the title as it has <br> in it and is not valid for file naming. 
    file_name = f"/FACS scatter_ {exp_name}" # The name of the file that is export in each loop.
    fig.write_image(r"C:\Users\ardal\Desktop\PhD project and Docs\2.Data\FACS 2023_ high and low uptakers exp\Exported graphs\scatterplots" + file_name + ".jpeg") # Export each subplot as an static image
    #fig.write_html(r"C:\Users\ardal\Desktop\facs_html" + file_name + ".html") # Export each subplot as an interactive plot that can be opened in a browser. 

    #fig.show() # remove hashtag to see the graph in the notebook
 
    #fig.show()

KeyError: 'Experimental_ID'

<div style="font-size: 0.6em; max-width: 70ch">

# Graph for the ratio of High to low

</div>

For each experiment Divide the median of High-uptke sample by the low-uptake sample and plot is using a lineplot

In [223]:
all_medians = [] 
for exp in medians["Experimental_ID"].unique():
    data = medians.loc[medians["Experimental_ID"] == exp]
 
 
 
    # As we only iloc we can perform mathematical calculation as it return an integer-based row, we first \ 
    # get the row based on the labels and then get the index of those rows for iloc. 
    row1 = data.loc[data["Sample"] == "Low"] 
    row2 = data.loc[data["Sample"] == "High"] 
    col = 'Size_normalized_Comp-YG-PE-A :: PKH26'
    
    # calculate the division of the two rows
    result = row2[col].values[0] / row1[col].values[0]

    # create a new row with the result of the division
    new_row = row2.copy()
    new_row[col] = result
  
    new_row["Sample"] = "High/Low ratio"
     
    data = pd.concat([data,new_row],ignore_index=True)
    data = data.loc[data["Sample"] == "High/Low ratio"]
    data.rename(columns={"Size_normalized_Comp-YG-PE-A :: PKH26": "PKH26_ratio"}, inplace=True)
    all_medians.append(data)
df_median_ratios = pd.concat(all_medians)

In [224]:
df_median_ratios.head(5)

,Experimental_ID,Sample,File name,PKH26_ratio,Time point,Empty
3,1,High/Low ratio,3jul2023 -PNT2C2 unsorted_PKH26 high- EV -DAPI...,1.530162,1,
3,2,High/Low ratio,12jul2023 -PNT2C2_PKH26 high- EV -DAPI_001_004...,4.285373,2,
3,3,High/Low ratio,4aug2023 -PNT2C2_PKH26 high- EV -DAPI_001_003.csv,15.873622,3,
3,4,High/Low ratio,11aug2023 -PNT2C2_PKH26 high- EV -DAPI_001_004...,16.054221,4,
3,5,High/Low ratio,21aug2023 -PNT2C2_PKH26 high- EV -DAPI_001_004...,17.489216,5,


In [233]:
fig = px.line(df_median_ratios, x="Time point" , y="PKH26_ratio",  text="Empty")

# Make a box around the figure and define the range of x and y axes.
fig.update_xaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", title_text="Time point", tick0= 1, dtick=1)
fig.update_yaxes(showline=True, linewidth=2, linecolor='black', mirror=True, ticks="outside", title_text="Median PKH26 signal intensity ratio <br> (High-uptake to Low-uptake)")

fig.update_layout(plot_bgcolor="white", title=dict(text="All FACS 2023 experiments over time", x=0.5))
 
fig.show()

In [ ]:
#data = df.loc[df["Experimental_ID"] == 2]

fig = px.histogram(df, x="Comp-YG-PE-A :: PKH26", color="Sample", nbins=256, animation_frame="Experimental_ID" , height=500, barmode="overlay", range_x=[0, 20000], range_y=[0, 3000], )
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 2000
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 4000
fig.update_geos(projection_type="equirectangular", visible=True, resolution=110)
# fig.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels=True)) # exponentformat="power",
# fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1])) # In the facet_col, the name of column is also included in the title. To remove that, we use this line to split \

fig.show()

<div style="font-size: 0.6em; max-width: 70ch">

# Extra

</div>

<div style="font-size: 0.6em; max-width: 70ch">

# Export the cleaned data.

</div>